# Train the transaction categorizer

TF-IDF on character 3-5-grams + Logistic Regression, used for payees the merchant rules don't recognise.

**Read this before quoting any score.** Training always starts from `data/labeled_transactions_synthetic.csv`, which is script-generated.
Scores on synthetic rows only show that the pipeline works. The numbers for your README and resume come from section 3,
which runs once your hand-labelled `data/raw/labeled_real.csv` exists.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import classification_report

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from core.training import build_pipeline, evaluate, group_split, load_real_labels

synthetic = pd.read_csv(ROOT / "data" / "labeled_transactions_synthetic.csv", dtype=str)
real = load_real_labels(ROOT / "data" / "raw" / "labeled_real.csv")
print(f"Synthetic rows: {len(synthetic)} | Real hand-labelled rows: {len(real)}")
synthetic["category"].value_counts()

Synthetic rows: 857 | Real hand-labelled rows: 0


category
Transfers (P2P)     155
Travel               96
Food & dining        95
Shopping             92
Subscriptions        90
Bills & recharge     83
Groceries            83
Education            82
Health               81
Name: count, dtype: int64

## 1. Check the pipeline on synthetic data

Every spelling of a merchant stays on one side of the split, so the model is tested on merchants it has never seen.

In [2]:
train, test = group_split(synthetic)
assert not set(train["merchant_group"]) & set(test["merchant_group"])

model = build_pipeline().fit(train["payee"], train["category"])
synthetic_scores = evaluate(model, test)
print(json.dumps(synthetic_scores, indent=2))

{
  "rules_only": {
    "accuracy": 0.285,
    "macro_f1": 0.409,
    "rows": 172
  },
  "rules_plus_ml": {
    "accuracy": 0.738,
    "macro_f1": 0.734,
    "rows": 172
  },
  "rule_coverage": 0.285
}


## 2. Where the model goes wrong (synthetic test rows)

In [3]:
print(classification_report(test["category"], model.predict(test["payee"]), zero_division=0))

                  precision    recall  f1-score   support

Bills & recharge       0.64      0.82      0.72        17
       Education       0.73      0.50      0.59        16
   Food & dining       0.60      0.63      0.62        19
       Groceries       0.36      0.62      0.45        16
          Health       0.75      0.75      0.75        16
        Shopping       0.31      0.22      0.26        18
   Subscriptions       0.73      0.44      0.55        18
 Transfers (P2P)       0.89      0.75      0.81        32
          Travel       0.62      0.75      0.68        20

        accuracy                           0.62       172
       macro avg       0.62      0.61      0.60       172
    weighted avg       0.65      0.62      0.62       172



## 3. Real evaluation on your hand-labelled transactions

20% of your real payees are held out and never used for training while scoring. Aim for 400-500 rows.

In [4]:
real_scores = None
if len(real) >= 50:
    real_train, real_test = group_split(real)
    real_model = build_pipeline().fit(
        pd.concat([synthetic, real_train])["payee"],
        pd.concat([synthetic, real_train])["category"],
    )
    real_scores = evaluate(real_model, real_test)
    print(json.dumps(real_scores, indent=2))
else:
    print(f"Skipped: {len(real)} real labelled rows so far; at least 50 needed. No real score exists yet.")

Skipped: 0 real labelled rows so far; at least 50 needed. No real score exists yet.


## 4. Train on everything and save

In [5]:
final = build_pipeline().fit(
    pd.concat([synthetic, real])["payee"],
    pd.concat([synthetic, real])["category"],
)
joblib.dump(final, ROOT / "models" / "categorizer.joblib", compress=3)

metrics = {
    "synthetic_test_not_for_reporting": synthetic_scores,
    "real_test": real_scores,
    "training_rows": {"synthetic": len(synthetic), "real": len(real)},
}
(ROOT / "models" / "categorizer_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
print("Saved models/categorizer.joblib with classes:", list(final.classes_))

Saved models/categorizer.joblib with classes: ['Bills & recharge', 'Education', 'Food & dining', 'Groceries', 'Health', 'Shopping', 'Subscriptions', 'Transfers (P2P)', 'Travel']
